This is just a simulation excersice to understand the attention mechanism

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

In [15]:
batch_size = 4
emb_dim = 10
context_len = 8
vocab_size = 40


In [ ]:
# Simulating the batched token data set
# torch.randint(max_value, (n_rows/dim1, n_cols/dim2))
data = torch.randint(vocab_size, (batch_size, context_len))
print(data.shape)

embeddings = nn.Embedding(vocab_size, emb_dim)

# For single head attention the size of K,Q,V is (emb_dim, emb_dim)
key = nn.Linear(emb_dim, emb_dim, bias=False)
query = nn.Linear(emb_dim, emb_dim, bias=False)
value = nn.Linear(emb_dim, emb_dim, bias=False)

torch.Size([4, 8])


In [ ]:
# Attention 
# A = softmax(
#           (q*k.T)/sqrt(k.shape[1]) + M
#       ) @ v

# generally A = softmax(QK.T/sqrt(dim_k) + M) @ v

x = embeddings(data)

# k,v,q are what goes into the attention equation
k = key(x)
v = value(x)
q = query(x)

print(f"Data matrix shape = {data.shape}")
print(f"Emb matrix shape = {embeddings.weight.shape}")
print(f"Token input embeddings shape = {x.shape}", "\n")

print(f"Size of Key matrix = {key.weight.shape}")
print(f"Size of Query matrix = {query.weight.shape}")
print(f"Size of Value matrix = {value.weight.shape}", "\n")

print(f"Size of k(x) = {k.shape}")
print(f"Size of q(x) = {q.shape}")
print(f"Size of v(x) = {v.shape}")



Data matrix shape = torch.Size([4, 8])
Emb matrix shape = torch.Size([40, 10])
Token input embeddings shape = torch.Size([4, 8, 10]) 

Size of Key matrix = torch.Size([10, 10])
Size of Query matrix = torch.Size([10, 10])
Size of Value matrix = torch.Size([10, 10]) 

Size of k(x) = torch.Size([4, 8, 10])
Size of q(x) = torch.Size([4, 8, 10])
Size of v(x) = torch.Size([4, 8, 10])


In [66]:
print("K shape = ", k.shape)
print("K.T shape = ", k.T.shape)
print("K.mT shape = ", k.mT.shape)
print("K.permute(0,2,1) shape = ", k.permute(0,2,1).shape)

P = k @ k.mT
print("P shape = ", P.shape)

K shape =  torch.Size([4, 8, 10])
K.T shape =  torch.Size([10, 8, 4])
K.mT shape =  torch.Size([4, 10, 8])
K.permute(0,2,1) shape =  torch.Size([4, 10, 8])
P shape =  torch.Size([4, 8, 8])


In [119]:
# Manual attention calc
# A = softmax(
#           (q*k.T)/sqrt(k.shape[1]) + M
#       ) @ v

# q => (4, 8, 10)
# k.mT => (4, 10, 8)
# q @ k.mT => (4, 8, 8)

qk = (q @ k.mT)/np.sqrt(k.shape[1])
print("Shape of qk = ", qk.shape)

# Mask
# Triangular lower matrix where only lower portions of matrix across the diagonal has vals and upper portion is 0
M = torch.tril(torch.ones(batch_size, context_len, context_len))
# print(qk[0])
masked_qk = qk * M # produces a triangular lower matrix where values same as qk tri lower, with upper tri portion as zeros
# print(masked_qk[0])
masked_qk[masked_qk == 0] = -torch.inf # Replace 0 with -inf. This equals "(q*k.T)/sqrt(k.shape[1]) + M"
# print(masked_qk[0]) 


# print(M)


softmaxed_qk = F.softmax(
    masked_qk, dim=2 # last dim, can be written as dim=-1
) 

print(torch.sum(softmaxed_qk, dim=-1))
# print(softmaxed_qk[0])

A_manual = softmaxed_qk @ v # (4, 8, 8) @ (4, 8, 10) => (4, 8, 10)
print("Size of acts = ", A_manual.shape)


Shape of qk =  torch.Size([4, 8, 8])
tensor([[1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000]],
       grad_fn=<SumBackward1>)
Size of acts =  torch.Size([4, 8, 10])


In [116]:
# pytorch implementation of attention algo
A = F.scaled_dot_product_attention(q, k ,v, is_causal=True)
print(A.shape)

print(A_manual[0][1], "\n")
print(A[0][1])

print(torch.allclose(A_manual, A, atol=1e-1)) # might print false but almost equal

torch.Size([4, 8, 10])
tensor([ 0.3758, -0.2607, -0.0880,  0.1676,  0.1007, -0.0605,  0.2795,  0.1583,
         0.1440,  0.1256], grad_fn=<SelectBackward0>) 

tensor([ 0.3756, -0.2609, -0.0878,  0.1675,  0.1005, -0.0607,  0.2799,  0.1580,
         0.1440,  0.1252], grad_fn=<SelectBackward0>)
True


In [19]:

import torch
so = torch.randint(0, 100, (1,8,4,12))
print(so)

yo = so.transpose(1,2)
print("="*50)
print(yo.shape)
print(yo)


tensor([[[[84, 57, 59, 98, 96, 18, 52, 33, 28, 28, 36, 77],
          [60, 34, 16, 44, 56, 29, 18, 70, 99, 50, 77, 33],
          [66, 99, 86, 21, 62, 84, 20, 27, 43, 56, 46, 38],
          [ 0, 35, 69, 94, 37, 62, 78, 92, 20, 30, 53, 12]],

         [[57, 92, 97,  3, 98, 66, 27, 72, 86, 35, 64, 53],
          [22, 24, 29, 52,  1,  8, 60,  4, 22,  9,  2, 85],
          [26, 52, 52, 56, 75, 55,  1, 41, 33, 83, 81, 16],
          [47, 25, 49, 49, 49,  7, 43, 39, 64, 61, 82, 64]],

         [[14, 19, 90, 41, 90, 20,  7, 97, 74, 13, 55, 96],
          [29, 55, 65, 93, 36, 83, 38,  3, 79, 40, 51, 12],
          [ 9,  8, 42, 31, 93, 47,  5, 21, 69, 19, 11, 88],
          [75, 70, 65, 41, 46,  7, 71, 23, 29, 56, 89, 49]],

         [[33, 10, 89, 42, 34, 65, 86, 83, 56, 75,  7, 52],
          [59, 90, 50,  4,  2,  5, 62, 46, 23, 79, 20, 97],
          [14, 33, 68, 34, 67, 40, 53, 58, 34, 76, 45, 14],
          [70, 96, 23, 91, 23, 18, 88, 37, 86, 27, 12,  8]],

         [[23, 64, 10, 93, 20, 5